In [1]:
from google.colab import drive
drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/melting_point_project"   # поправьте под своё название папки
RAW = f"{BASE}/data/raw"
PROC = f"{BASE}/data/processed"

Mounted at /content/drive


In [2]:
import pandas as pd

In [19]:
df = pd.read_excel(f"{RAW}/BradleyMeltingPointDataset.xlsx")


In [3]:
X = pd.read_parquet(f"{PROC}/descriptors.parquet")      # или pd.read_csv(...)
meta = pd.read_csv(f"{PROC}/melting_points_clean.csv")
y = meta["mpC"].values
print(X.shape, len(y))

(19824, 215) 19824


In [4]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

def report(name, pred):
  mae = mean_absolute_error(y_test, pred)
  rmse = mean_squared_error(y_test, pred) ** 0.5
  print(f"{name}: MAE = {mae:.2f}, RMSE = {rmse:.2f}")

report('median', np.full(len(y_test), np.median(y_train)))

median: MAE = 72.54, RMSE = 91.40


In [5]:
model = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestRegressor(n_estimators=50,n_jobs=-1, random_state=0)
)
model.fit(X_train, y_train)
report('RF', model.predict(X_test))

RF: MAE = 29.43, RMSE = 39.58


In [24]:
gain = 1 - 39.28 / 96.61
print(f" RMSE: {gain:.0%}")

улучшение RMSE: 59%
